[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **Egocentric Cooking Step Recipe Detector**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)

## Project Overview  
POV Cooking Step Timeline Detector is an AI-powered computer vision system designed to automatically analyze and segment first-person culinary videos into structured cooking timelines.

The system combines:
- Global cooking scene classification  
- Fine-grained hand keypoint tracking  

to detect cooking phases such as:
- Ingredient Preparation  
- Mixing & Blending  
- Thermal Preparation  
- Active Cooking  
- Seasoning  
- Plating  

By integrating scene understanding with hand-motion analysis, the model generates a continuous and intelligent timeline directly from raw POV cooking footage.

---

## Core Architecture  

### Global Scene Recognition  
Identifies the current cooking phase based on the full kitchen environment.

### Hand Keypoint Tracking  
Tracks detailed hand movements using skeletal landmark estimation.

### Temporal Timeline Mapping  
Combines scene transitions and hand actions to build a chronological cooking workflow.

### Structured Video Understanding  
Transforms unstructured cooking footage into searchable and segmented procedural data.

---

## Real-World Applications  

### Smart Kitchen Assistants  
Automatically advances recipe instructions without requiring touch interaction.

### Automated Recipe Editing  
Generates timestamps and chapter markers for cooking content creators.

### Commercial Kitchen Analytics  
Monitors workflow consistency, timing, and cooking procedure compliance.

### Culinary Training  
Evaluates cooking techniques and hand coordination for skill assessment.

### Video Search & Indexing  
Allows direct navigation to moments like seasoning, plating, or mixing within long cooking videos.

---

## Annotate your Custom dataset using Labellerr

 ***1. Visit the [Labellerr](https://www.labellerr.com/?utm_source=githubY&utm_medium=social&utm_campaign=github_clicks) website and click **“Sign Up”**.*** 

 ***2. After signing in, create your workspace by entering a unique name.***

 ***3. Navigate to your workspace’s API keys page (e.g., `https://<your-workspace>.labellerr.com/workspace/api-keys`) to generate your **API Key** and **API Secret**.***

 ***4. Store the credentials securely, and then use them to initialise the SDK or API client with `api_key`, `api_secret`.*** 



## Import Libraries

This section imports all the required libraries used throughout the project for computer vision, visualization, deep learning, and structured coding.


In [1]:
!git clone https://github.com/Labellerr/yolo_finetune_utils.git

Cloning into 'yolo_finetune_utils'...


In [2]:
!pip install ultralytics opencv-python matplotlib cv2

ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)

[notice] A new release of pip is available: 24.3.1 -> 26.1.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for cv2


## 📥 Download Annotations from Labellerr

After completing data labeling on the **Labellerr** platform, export the annotations in **COCO JSON format**.

Download the COCO JSON file from the Labellerr website and upload it into this project workspace to use it for further dataset preparation and training.

This COCO JSON file will be used in the next steps for:
- Frame–annotation alignment
- COCO → YOLO format conversion
- Model training and evaluation


# COCO to YOLO Format Conversion

Converts COCO-style segmentation annotations to YOLO segmentation dataset format.  
- Requires: `annotation.json` and images in `frames_output` directory.
- Output: Generated YOLO dataset folder.
- Parameters: allows train/val split, shuffling, and verbose mode.


In [5]:
import os
import cv2
import json
import random

# --- CONFIGURATION ---
VIDEO_PATH = "pov_omlete_making.mp4"
JSON_PATH = "export-#6bf41ee5-d187-4345-b1eb-f44cc9610cd0.json"
OUTPUT_DIR = "kitchen_dataset"
FRAME_STRIDE = 3  # Save every 3rd frame to prevent redundancy

# 1. Load the Labellerr JSON structure
with open(JSON_PATH, "r") as f:
    raw_data = json.load(f)

# The video asset blocks sit inside the first element array
video_entry = raw_data[0]
latest_answers = video_entry.get("file_metadata", {})
video_fps = latest_answers.get("fps", 23)  # Pulls the 23 FPS straight from your file metadata

# 2. Extract and parse event segments from answers
parsed_events = []
for question_block in video_entry.get("latest_answer", []):
    event_label = question_block.get("question_name").replace(" ", "_")
    
    # Iterate through all occurrences of this event
    for segment in question_block.get("answer", []):
        start_frame = int(segment.get("startFrame"))
        end_frame = int(segment.get("endFrame"))
        
        parsed_events.append({
            "label": event_label,
            "start_frame": start_frame,
            "end_frame": end_frame
        })

print(f"Successfully extracted {len(parsed_events)} event segments from Labellerr JSON.")

# 3. Dynamically build YOLO class classification folders
splits = ['train', 'val']
for split in splits:
    for event in parsed_events:
        os.makedirs(os.path.join(OUTPUT_DIR, split, event["label"]), exist_ok=True)

# 4. Process the Video Frames via Frame Index Tracking
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

print(f"Processing video at {video_fps} FPS...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Check if the current frame matches any parsed event boundaries
    assigned_event = None
    for event in parsed_events:
        if event["start_frame"] <= frame_count <= event["end_frame"]:
            assigned_event = event["label"]
            break  # Break out early if found
            
    if assigned_event and (frame_count % FRAME_STRIDE == 0):
        # 80/20 train/validation assignment
        split = "train" if random.random() < 0.80 else "val"
        
        frame_name = f"frame_{frame_count:06d}.jpg"
        save_path = os.path.join(OUTPUT_DIR, split, assigned_event, frame_name)
        cv2.imwrite(save_path, frame)
            
    frame_count += 1

cap.release()
print(f"🚀 YOLO Classification Dataset compiled successfully inside: '{OUTPUT_DIR}/'")

Successfully extracted 7 event segments from Labellerr JSON.
Processing video at 23 FPS...
🚀 YOLO Classification Dataset compiled successfully inside: 'kitchen_dataset/'


# Load and Train YOLO Pose Model

Loads the YOLO segmentation model and trains it using the converted YOLO dataset.
- Data: Path to YOLO-style `data.yaml`
- Parameters: epochs, image size, batch size, device, dataloader workers, experiment name.


In [ ]:
from ultralytics import YOLO

# Load a lightweight pre-trained classification model
model = YOLO("yolo11n-cls.pt") 

# Train on your custom kitchen event frames
results = model.train(
    data="kitchen_dataset", 
    epochs=50,             
    imgsz=640,             
    batch=16,              
    device='cpu'  # <--- Change this to run on CPU
)

# YOLO-Pose Format Conversion

## Overview  
Converts Labellerr-style hand keypoint annotations into the standard **YOLO-Pose** dataset format for pose estimation training.

---

## Requirements  

- Raw hand keypoint JSON annotations  
- Corresponding source video file  

---

## Output  

Generates a structured dataset folder:

`hand_pose_dataset/`

Containing:
- Extracted image frames  
- YOLO-Pose normalized label files  
- Train/validation directory split  

---

In [13]:
import os
import cv2
import json
import random

# --- CONFIGURATION ---
VIDEO_PATH = "pov_omlete_making.mp4"
JSON_PATH = "c330dd92-83d5-413d-b028-8f4e450c9547 (2).json"
OUTPUT_DIR = "hand_pose_dataset"

# 1. Initialize directory layout for YOLO Pose tracking
for split in ['train', 'val']:
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split, 'labels'), exist_ok=True)

# 2. Parse Labellerr raw JSON coordinates
with open(JSON_PATH, "r") as f:
    raw_data = json.load(f)

# FIX: Read directly from the dictionary root instead of raw_data[0]
video_entry = raw_data
metadata = video_entry.get("file_metadata", {})
img_w = metadata.get("width", 1280)
img_h = metadata.get("height", 720)

# Dictionary to hold frames grouped by frame index to handle multiple hands per frame
frame_annotations = {}

for question_block in video_entry.get("latest_answer", []):
    # Process blocks containing hand coordinate mappings
    if question_block.get("question_type") == "keypoint" or "keypoint" in str(question_block.keys()):
        for item in question_block.get("answer", []):
            # Fallback handling for frame keys
            frame_idx = item.get("frameIndex")
            if frame_idx is None:
                frame_idx = item.get("startFrame", 0)
            frame_idx = int(frame_idx)
            
            # Extract bounding box parameters
            bbox = item.get("bbox", {})
            bx = bbox.get("x")
            by = bbox.get("y")
            bw = bbox.get("width")
            bh = bbox.get("height")
            
            # Check for edge cases where a bounding box might be missing or unannotated
            if bx is None or by is None or bw is None or bh is None:
                continue
                
            # Calculate standard YOLO normalized box parameters
            x_center = (bx + (bw / 2)) / img_w
            y_center = (by + (bh / 2)) / img_h
            norm_w = bw / img_w
            norm_h = bh / img_h
            
            # Extract keypoint coordinate vectors
            keypoints_list = item.get("keypoints", item.get("points", []))
            normalized_kpts = []
            for kp in keypoints_list:
                kx = kp.get("x")
                ky = kp.get("y")
                # Normalize points
                normalized_kpts.append(f"{kx / img_w:.6f} {ky / img_h:.6f}")
            
            kpts_str = " ".join(normalized_kpts)
            
            # Construct line payload (Class index 0 for hand instance)
            yolo_line = f"0 {x_center:.6f} {y_center:.6f} {norm_w:.6f} {norm_h:.6f} {kpts_str}\n"
            
            if frame_idx not in frame_annotations:
                frame_annotations[frame_idx] = []
            frame_annotations[frame_idx].append(yolo_line)

# 3. Process video, extract keyframes, and output files
cap = cv2.VideoCapture(VIDEO_PATH)
frame_count = 0

print(f"Converting annotations. Exporting to '{OUTPUT_DIR}' directory...")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # Only process frames that have annotated keypoints
    if frame_count in frame_annotations:
        # Determine 80/20 partition split group
        split = "train" if random.random() < 0.80 else "val"
        
        base_name = f"frame_{frame_count:06d}"
        
        # Save image frame asset
        img_path = os.path.join(OUTPUT_DIR, split, 'images', f"{base_name}.jpg")
        cv2.imwrite(img_path, frame)
        
        # Save matching annotation lines text file
        label_path = os.path.join(OUTPUT_DIR, split, 'labels', f"{base_name}.txt")
        with open(label_path, "w") as lf:
            lf.writelines(frame_annotations[frame_count])
            
    frame_count += 1

cap.release()
print(f"🚀 Dataset conversion complete! Labels generated successfully inside '{OUTPUT_DIR}/'")

Converting annotations. Exporting to 'hand_pose_dataset' directory...
🚀 Dataset conversion complete! Labels generated successfully inside 'hand_pose_dataset/'


# YOLO11-Pose Model Training

- Fine-tunes a custom **YOLO11-Pose** model using `hand_pose.yaml` and `yolo11s-pose.pt`.  
- Trains for **75 epochs** at **640px** resolution using GPU acceleration (`device=0`).  
- Uses **4 workers** for faster batch loading.  
- Saves optimized weights (`best.pt`, `last.pt`) to:

`runs/pose/train/weights/`

---

In [ ]:
from ultralytics import YOLO

# Load pre-trained weights engineered for Pose keypoint estimation
model = YOLO("yolo11s-pose.pt")

# Train the model on your custom kitchen hand landmarks
results = model.train(
    data="hand_pose.yaml",
    epochs=75,             # 75 epochs is a good baseline for tuning fine-grained coordinates
    imgsz=640,             # Keep standard inference image dimensions
    batch=16,              # Adjust if you run into GPU memory constraints
    device=0,              # Runs natively on your GPU
    workers=4              # Adjust based on your CPU threads for fast data loading
)

# Dual YOLO11 Inference Pipeline

- Loads two custom YOLO11 models:
  - Event Classification Model  
  - Hand Pose Estimation Model  

- Performs:
  - Global cooking step recognition  
  - Real-time hand keypoint detection  

- Uses native YOLO pose rendering for skeleton visualization.

- Displays a clean black dashboard showing the current cooking stage:
  - Prep Ingredients  
  - Mixing Blending  
  - Thermal Prep  
  - Active Cooking  
  - Seasoning  
  - Plating  

- Processes video frame-by-frame with real-time preview support.

---

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

# --- CONFIGURATION ---
VIDEO_PATH = "pov_omlete_making.mp4"
CLASSIFY_MODEL_PATH = "best.pt"  # Your Event model
POSE_MODEL_PATH = "best (1).pt"          # Your Hand Pose model

# 1. Load both custom trained models
print("Loading custom YOLO11 models...")
event_model = YOLO(CLASSIFY_MODEL_PATH)
hand_pose_model = YOLO(POSE_MODEL_PATH)

# Define your 6 event clean display mappings
EVENT_DISPLAY_NAMES = {
    "Prep_Ingredients": "Prep Ingredients",
    "Mixing_Blending": "Mixing Blending",
    "Thermal_Prep": "Thermal Prep",
    "Active_Cooking": "Active Cooking",
    "Seasoning": "Seasoning",
    "Plating": "Plating"
}

# 2. Open Video Stream
cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file {VIDEO_PATH}")
    exit()

# Get screen resolution variables for rendering safeguards
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("Starting simultaneous inference loop. Press 'q' to exit.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
        
    # --- STEP 1: GLOBAL EVENT TAGGING ---
    event_results = event_model(frame, verbose=False)
    raw_class_name = event_results[0].names[event_results[0].probs.top1]
    
    # Clean up the name mapping for display
    display_status = EVENT_DISPLAY_NAMES.get(raw_class_name, raw_class_name)
    status_text = f"status: {display_status.lower()}"
    
    # --- STEP 2: LOCAL HAND KEYPOINT DETECTION ---
    pose_results = hand_pose_model(frame, verbose=False)
    
    # Draw keypoints and skeletons natively if any hands are found
    if pose_results[0].keypoints is not None:
        frame = pose_results[0].plot()  # Natively plots boxes and skeleton linkages
        
    # --- STEP 3: RENDER BLACK BOX TEXT OVERLAY ---
    # Setup text sizing configurations dynamically to fit the background box
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.8
    thickness = 2
    text_color = (255, 255, 255)  # White text
    
    # Calculate exact pixels needed for the text boundaries
    (text_w, text_h), baseline = cv2.getTextSize(status_text, font, font_scale, thickness)
    
    # Define box dimensions padding metrics
    padding_x = 15
    padding_y = 15
    box_x1 = 10
    box_y1 = 10
    box_x2 = box_x1 + text_w + (padding_x * 2)
    box_y2 = box_y1 + text_h + (padding_y * 2)
    
    # Render the opaque black bounding background rectangle box
    cv2.rectangle(frame, (box_x1, box_y1), (box_x2, box_y2), (0, 0, 0), cv2.FILLED)
    
    # Calculate text origin point coordinate line offset anchor
    text_org_x = box_x1 + padding_x
    text_org_y = box_y1 + text_h + padding_y
    
    # Write text cleanly over the black box region
    cv2.putText(frame, status_text, (text_org_x, text_org_y), font, font_scale, text_color, thickness, cv2.LINE_AA)
    
    # --- STEP 4: PREVIEW DISPLAY WINDOW ---
    # Optional window resize safety boundary in case frame dimensions exceed monitors
    cv2.imshow("Cooking Step Detector - Real-time Pipeline", frame)
    
    # Break frame process loop safely if user triggers 'q' key threshold escape
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Pipeline execution stopped successfully.")

---

## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
